# AutoGen Notes: Multi-Agent Systems

> Note: AutoGen has evolved significantly from older `autogen` / `pyautogen` examples to the newer packages such as `autogen-agentchat`, `autogen-core`, and `autogen-ext`. Many older tutorials use `AssistantAgent`, `UserProxyAgent`, and `GroupChat` from older APIs. This notebook explains both the concepts and practical patterns, with runnable examples that do not require paid API keys for the simulation sections.

## 1. What is a Multi-Agent System?

A **Multi-Agent System (MAS)** is an application where multiple independent or semi-independent agents collaborate to solve a task.

In GenAI, an **agent** usually means a component that can:

1. receive a goal or message,
2. reason about what to do,
3. optionally call tools,
4. produce a response or action,
5. collaborate with other agents.

### Simple analogy
Imagine building an e-commerce website:

- **Product Manager Agent** clarifies requirements.
- **Backend Developer Agent** designs APIs.
- **Frontend Developer Agent** designs pages.
- **QA Agent** writes test cases.
- **Security Agent** checks authentication and data protection.
- **Reviewer Agent** validates the final plan.

Instead of one LLM trying to do everything, we split the work across specialized agents.

## 2. Why do we need multiple agents?

A single LLM can answer many questions, but complex tasks often need different skills.

### Benefits

| Benefit | Meaning | Example |
|---|---|---|
| Specialization | Each agent focuses on one role | QA agent writes test cases; Security agent checks risks |
| Better reasoning | Agents critique each other | Reviewer catches missing edge cases |
| Tool separation | Different agents can use different tools | Data agent queries DB; Code agent writes Python |
| Human-like workflow | Mirrors real project teams | BA → Developer → QA → Reviewer |
| Reusability | Same agent can be reused | Same QA agent across many projects |

### Risks

| Risk | What can go wrong |
|---|---|
| Cost | Multiple agents may call the LLM many times |
| Latency | More conversation rounds can be slow |
| Hallucination propagation | One wrong answer can influence others |
| Infinite loops | Agents may keep talking without finishing |
| Poor orchestration | Wrong agent order can reduce quality |

## 3. Core building blocks of a GenAI Multi-Agent System

A practical multi-agent system usually contains:

```text
User Goal
   ↓
Orchestrator / Team Manager
   ↓
Agent 1 ─ Agent 2 ─ Agent 3
   ↓        ↓         ↓
Tools    Memory    External APIs
   ↓
Final Answer / Action
```

### Key terms

| Term | Meaning |
|---|---|
| Agent | A role-based AI worker, e.g., Planner, Coder, QA |
| Message | Communication between agents |
| Tool | Function/API/database/code executor used by an agent |
| Orchestrator | Controls which agent acts next |
| Team | Group of agents following a pattern |
| Termination condition | Rule that stops the conversation |
| Human-in-the-loop | Human approval or input during workflow |

## 4. What is AutoGen?

**AutoGen** is a framework originally developed by Microsoft for building AI agents and multi-agent applications. It supports agent conversations, tool usage, code execution, and team-based workflows.

Modern AutoGen is commonly organized around:

| Layer | Purpose |
|---|---|
| AgentChat | High-level API for building conversational single-agent and multi-agent apps |
| Core | Lower-level event-driven framework for scalable systems |
| Extensions | Integrations with model providers, tools, code executors, MCP, etc. |
| Studio | UI-based prototyping and debugging experience |

### Important practical note
Older examples on the internet may use APIs like:

```python
import autogen
assistant = autogen.AssistantAgent(...)
user_proxy = autogen.UserProxyAgent(...)
```

Newer examples commonly use packages like:

```python
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
```

So always check the version of AutoGen used in your course.

## 5. Common Multi-Agent Design Patterns

### Pattern 1: Sequential workflow
Agents work one after another.

```text
Planner → Developer → Tester → Reviewer → Final Output
```

Best for predictable tasks such as:

- creating test strategy,
- writing a report,
- generating code and tests,
- summarizing documents step by step.

### Pattern 2: Round-robin discussion
Agents take turns until a stop condition is reached.

```text
Agent A → Agent B → Agent C → Agent A → ... → STOP
```

Best for brainstorming, review, and collaborative refinement.

### Pattern 3: Manager / supervisor pattern
A manager decides which agent should work next.

```text
Manager
 ├── Research Agent
 ├── Code Agent
 ├── QA Agent
 └── Security Agent
```

Best for dynamic tasks where the next step is not fixed.

### Pattern 4: Critic / reviewer pattern
One agent produces output and another critiques it.

```text
Writer → Critic → Writer improves → Final
```

Best for improving quality and reducing mistakes.

### Pattern 5: Tool-specialist pattern
Different agents own different tools.

```text
SQL Agent → Database
Python Agent → Code execution
Web Agent → Search
QA Agent → Validation
```

# Part A: Runnable Simulation Without LLM API Key

The next examples simulate multi-agent behavior using normal Python classes. This helps you understand the architecture before using real LLM-based AutoGen agents.

In [64]:
from dataclasses import dataclass
from typing import List, Callable, Dict

@dataclass
class Message:
    sender: str
    content: str

class SimpleAgent:
    """A small non-LLM agent used to demonstrate multi-agent flow."""
    def __init__(self, name: str, role: str, behavior: Callable[[List[Message]], str]):
        self.name = name
        self.role = role
        self.behavior = behavior

    def respond(self, conversation: List[Message]) -> Message:
        reply = self.behavior(conversation)
        return Message(sender=self.name, content=reply)

def print_conversation(conversation: List[Message]):
    for msg in conversation:
        print(f"\n[{msg.sender}]\n{msg.content}")

## 6. Example 1: Sequential agents for an e-commerce feature

Real-world scenario: You want to build a **Product Search** feature for your saree / kurti e-commerce website.

Agents:

1. **Business Analyst Agent** clarifies requirements.
2. **Backend Agent** designs API.
3. **QA Agent** writes test cases.
4. **Reviewer Agent** identifies missing points.

In [10]:
def ba_behavior(conversation):
    return """
Requirement summary:
- User should search products by keyword such as 'silk saree' or 'cotton kurti'.
- User should filter by category, price range, color, size, fabric, and availability.
- Results should show product image, name, price, discount, and stock status.
- Search should work on mobile and desktop.
""".strip()

def backend_behavior(conversation):
    return """
Backend API design:
GET /api/products/search
Query parameters:
- q: search keyword
- category: saree/kurti/blouse
- min_price, max_price
- color, size, fabric
- in_stock: true/false

Response:
{
  "items": [...],
  "total": 120,
  "page": 1,
  "page_size": 20
}
""".strip()

def qa_behavior(conversation):
    return """
Test cases:
1. Search with valid keyword returns relevant products.
2. Search with no matching keyword shows 'No products found'.
3. Filter by price range returns products within range.
4. Filter by size should not show out-of-stock sizes.
5. Search response time should be acceptable for large catalog.
6. Verify mobile UI displays image, price, and discount correctly.
""".strip()

def reviewer_behavior(conversation):
    return """
Review comments:
- Add sorting: price low to high, newest, popularity.
- Add pagination test cases.
- Add security checks for invalid query parameters.
- Add analytics tracking for searched keywords.
""".strip()

agents = [
    SimpleAgent("BusinessAnalystAgent", "Clarifies business requirements", ba_behavior),
    SimpleAgent("BackendAgent", "Designs backend APIs", backend_behavior),
    SimpleAgent("QAAgent", "Creates test cases", qa_behavior),
    SimpleAgent("ReviewerAgent", "Reviews gaps", reviewer_behavior),
]

conversation = [Message("User", "Design product search for my e-commerce website.")]

for agent in agents:
    conversation.append(agent.respond(conversation))

print_conversation(conversation)


[User]
Design product search for my e-commerce website.

[BusinessAnalystAgent]
Requirement summary:
- User should search products by keyword such as 'silk saree' or 'cotton kurti'.
- User should filter by category, price range, color, size, fabric, and availability.
- Results should show product image, name, price, discount, and stock status.
- Search should work on mobile and desktop.

[BackendAgent]
Backend API design:
GET /api/products/search
Query parameters:
- q: search keyword
- category: saree/kurti/blouse
- min_price, max_price
- color, size, fabric
- in_stock: true/false

Response:
{
  "items": [...],
  "total": 120,
  "page": 1,
  "page_size": 20
}

[QAAgent]
Test cases:
1. Search with valid keyword returns relevant products.
2. Search with no matching keyword shows 'No products found'.
3. Filter by price range returns products within range.
4. Filter by size should not show out-of-stock sizes.
5. Search response time should be acceptable for large catalog.
6. Verify mobile

### Learning from Example 1

This is a **sequential multi-agent workflow**.

The output quality improves because each agent contributes a different perspective:

- BA focuses on user/business needs.
- Backend focuses on API design.
- QA focuses on validation.
- Reviewer finds gaps.

This is similar to how real software teams work.

## 7. Example 2: Critic pattern for improving a test strategy

In GenAI systems, a critic agent is useful because the first answer from an LLM may miss edge cases.

Here, one agent writes a test strategy and another agent critiques it.

In [13]:
def test_strategy_writer(conversation):
    return """
Initial test strategy for login:
- Verify login with valid email and password.
- Verify error message for invalid password.
- Verify forgot password link.
- Verify logout works.
""".strip()

def critic_agent(conversation):
    last_output = conversation[-1].content
    return f"""
Critique of test strategy:
The strategy is good but incomplete.
Missing areas:
- Account lock after repeated failed attempts.
- Two-factor authentication.
- SQL injection / XSS input checks.
- Session timeout.
- Remember me functionality.
- Login from different browsers/devices.
- API-level login validation.
""".strip()

def improved_writer(conversation):
    return """
Improved login test strategy:
Functional:
- Valid login, invalid password, invalid email, blank fields.
- Forgot password and reset password flow.
- Remember me and logout.

Security:
- Account lock after failed attempts.
- 2FA verification.
- SQL injection and XSS input validation.
- Session timeout and token expiry.

Compatibility:
- Chrome, Edge, Firefox, mobile browser.

API:
- Validate status codes: 200, 400, 401, 403, 429.
- Validate response body and token format.
""".strip()

writer = SimpleAgent("WriterAgent", "Writes first draft", test_strategy_writer)
critic = SimpleAgent("CriticAgent", "Finds gaps", critic_agent)
improver = SimpleAgent("ImproverAgent", "Improves based on critique", improved_writer)

conversation = [Message("User", "Create a login test strategy.")]
for agent in [writer, critic, improver]:
    conversation.append(agent.respond(conversation))

print_conversation(conversation)


[User]
Create a login test strategy.

[WriterAgent]
Initial test strategy for login:
- Verify login with valid email and password.
- Verify error message for invalid password.
- Verify forgot password link.
- Verify logout works.

[CriticAgent]
Critique of test strategy:
The strategy is good but incomplete.
Missing areas:
- Account lock after repeated failed attempts.
- Two-factor authentication.
- SQL injection / XSS input checks.
- Session timeout.
- Remember me functionality.
- Login from different browsers/devices.
- API-level login validation.

[ImproverAgent]
Improved login test strategy:
Functional:
- Valid login, invalid password, invalid email, blank fields.
- Forgot password and reset password flow.
- Remember me and logout.

Security:
- Account lock after failed attempts.
- 2FA verification.
- SQL injection and XSS input validation.
- Session timeout and token expiry.

Compatibility:
- Chrome, Edge, Firefox, mobile browser.

API:
- Validate status codes: 200, 400, 401, 403,

### Learning from Example 2

The critic pattern is powerful for:

- test case improvement,
- resume improvement,
- code review,
- architecture review,
- security review,
- document quality checks.

But it can increase cost because each extra review step may require another LLM call.

## 8. Example 3: Tool-using agent pattern

Agents become more useful when they can call tools.

A tool can be:

- Python function,
- database query,
- API call,
- file reader,
- search function,
- calculator,
- code executor.

Below, we create a simple inventory-checking tool and let an agent use it.

In [16]:
# Sample inventory data for an e-commerce store
inventory = {
    "silk_saree_red": {"stock": 4, "price": 3500},
    "cotton_saree_blue": {"stock": 0, "price": 1200},
    "kurti_black_m": {"stock": 12, "price": 899},
}

def check_inventory(product_id: str) -> str:
    item = inventory.get(product_id)
    if not item:
        return f"Product '{product_id}' not found."
    if item["stock"] <= 0:
        return f"{product_id} is out of stock."
    return f"{product_id} is available. Stock={item['stock']}, Price=₹{item['price']}"

class InventoryAgent:
    def __init__(self, name="InventoryAgent"):
        self.name = name

    def respond(self, product_id: str) -> Message:
        result = check_inventory(product_id)
        return Message(self.name, result)

agent = InventoryAgent()
for product in ["silk_saree_red", "cotton_saree_blue", "unknown_product"]:
    msg = agent.respond(product)
    print(f"[{msg.sender}] {msg.content}")

[InventoryAgent] silk_saree_red is available. Stock=4, Price=₹3500
[InventoryAgent] cotton_saree_blue is out of stock.
[InventoryAgent] Product 'unknown_product' not found.


### Learning from Example 3

This is the basic idea behind tool calling in agent frameworks.

The LLM should not guess inventory. It should call the inventory tool and answer using real data.

In production, the tool may connect to:

- PostgreSQL,
- Snowflake,
- REST API,
- GCP services,
- internal enterprise systems.

# Part B: AutoGen-style Practical Notes

The next sections show how you would use AutoGen-style APIs. Some cells are designed as templates because they require package installation and API keys.

## 9. Installing AutoGen packages

Run this in your terminal or notebook environment:

```bash
pip install -U autogen-agentchat autogen-core "autogen-ext[openai]"
```

For AutoGen Studio:

```bash
pip install -U autogenstudio
autogenstudio ui --port 8080 --appdir ./myapp
```

### Python version
Use Python 3.10 or later.

In [53]:
# Optional: check Python version
import sys
print(sys.version)

from dotenv import load_dotenv
load_dotenv()

3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]


True

## 10. AutoGen AgentChat: single assistant example

This example shows the basic structure of an AutoGen assistant.

> This cell is a template. It needs an API key and installed AutoGen packages.

In [56]:
# Template only: requires autogen-agentchat, autogen-ext[openai], and OPENAI_API_KEY

import os
import asyncio

async def run_single_autogen_agent():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_ext.models.openai import OpenAIChatCompletionClient
    except ImportError:
        print("AutoGen packages not installed. Run: pip install -U autogen-agentchat 'autogen-ext[openai]'")
        return

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found. Set it before running this example.")
        return

    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    assistant = AssistantAgent(
        name="genai_trainer",
        model_client=model_client,
        system_message="You are a helpful GenAI trainer. Explain concepts with simple real-world examples."
    )

    response = await assistant.run(task="Explain multi-agent systems with an e-commerce example.")
    print(response)

# In Jupyter, run:
await run_single_autogen_agent()

messages=[TextMessage(id='aedbd5d7-ed14-4467-b4bf-15656ce59fce', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 42, 16, 830509, tzinfo=datetime.timezone.utc), content='Explain multi-agent systems with an e-commerce example.', type='TextMessage'), TextMessage(id='0b1872e3-d3dd-4c8f-b542-63254c84a174', source='genai_trainer', models_usage=RequestUsage(prompt_tokens=38, completion_tokens=499), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 42, 26, 851526, tzinfo=datetime.timezone.utc), content='Sure! A multi-agent system consists of multiple agents that interact and collaborate to achieve specific goals. Each agent can act independently, but they also communicate and coordinate with one another.\n\nLet’s break this down with a simple e-commerce example:\n\n**Scenario**: Imagine an online shopping platform where various agents are working together.\n\n### Agents in the System:\n1. **Customer Agent**: This represents the shopper. It tr

## 11. AutoGen multi-agent team: RoundRobinGroupChat

A **RoundRobinGroupChat** lets agents speak one by one in a fixed order.

Real-world use case:

```text
PlannerAgent → DeveloperAgent → QAAgent → ReviewerAgent
```

The team stops when a termination condition is reached, such as:

- maximum number of messages,
- keyword like `APPROVED`,
- human approval,
- task completion signal.

In [61]:
# Template only: requires AutoGen packages and API key

import os
import asyncio

async def run_round_robin_team():
    try:
        from autogen_agentchat.agents import AssistantAgent
        from autogen_agentchat.teams import RoundRobinGroupChat
        from autogen_agentchat.conditions import MaxMessageTermination
        from autogen_ext.models.openai import OpenAIChatCompletionClient
    except ImportError:
        print("AutoGen packages not installed.")
        return

    if not os.getenv("OPENAI_API_KEY"):
        print("OPENAI_API_KEY not found.")
        return

    model_client = OpenAIChatCompletionClient(
        model="gpt-4o-mini",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    planner = AssistantAgent(
        "planner_agent",
        model_client=model_client,
        system_message="You are a business analyst. Break the task into clear requirements."
    )

    developer = AssistantAgent(
        "developer_agent",
        model_client=model_client,
        system_message="You are a backend developer. Suggest API design and implementation approach."
    )

    qa = AssistantAgent(
        "qa_agent",
        model_client=model_client,
        system_message="You are a senior QA engineer. Create functional, API, and edge test cases."
    )

    reviewer = AssistantAgent(
        "reviewer_agent",
        model_client=model_client,
        system_message="You are a strict reviewer. Find gaps and approve only when complete."
    )

    termination = MaxMessageTermination(max_messages=8)

    team = RoundRobinGroupChat(
        participants=[planner, developer, qa, reviewer],
        termination_condition=termination
    )

    result = await team.run(task="Design a product search feature for an e-commerce saree website.")
    print(result)

# In Jupyter, run:
await run_round_robin_team()

messages=[TextMessage(id='d0d8349b-a8f3-4399-a647-2ea98b01f8ef', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 42, 44, 741770, tzinfo=datetime.timezone.utc), content='Design a product search feature for an e-commerce saree website.', type='TextMessage'), TextMessage(id='6d53dff9-d0e0-48a6-8c6c-ebdf984e1ad0', source='planner_agent', models_usage=RequestUsage(prompt_tokens=38, completion_tokens=720), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 42, 54, 612047, tzinfo=datetime.timezone.utc), content='To design a product search feature for an e-commerce saree website, we need to outline clear requirements that encompass functionality, user experience, and technical specifications. Below are the key requirements categorized by different aspects.\n\n### 1. Functional Requirements\n\n#### 1.1 Basic Search Functionality\n- **Search Bar**: A prominently placed search bar on the homepage and other relevant pages for easy accessibility.\n

To design a product search feature for an e-commerce saree website, we need to outline clear requirements that encompass functionality, user experience, and technical specifications. Below are the key requirements categorized by different aspects.\n\n### 1. Functional Requirements\n\n#### 1.1 Basic Search Functionality\n- **Search Bar**: A prominently placed search bar on the homepage and other relevant pages for easy accessibility.\n- **Keyword Matching**: Ability to search products by keywords (e.g., "silk saree", "handwoven saree").\n- **Search Button**: A button to initiate the search process after entering keywords.\n\n#### 1.2 Advanced Search Options\n- **Filters**: Provide filter options for users to narrow down their search results based on attributes such as:\n  - Category (e.g., party wear, casual, bridal)\n  - Fabric (e.g., cotton, silk)\n  - Color (e.g., red, blue, multi-color)\n  - Price range\n  - Brand\n  - Size\n- **Sort by Options**: Users should be able to sort results by relevance, price (low to high or high to low), newest arrivals, etc.\n\n#### 1.3 Autocomplete and Suggestions\n- **Autocomplete**: Suggest relevant search queries as users type in the search bar.\n- **Popular Searches**: Display a list of popular search queries to guide users.\n- **Related Products**: Show related products based on the current search term.\n\n### 2. User Experience Requirements\n\n#### 2.1 Responsive Design\n- Ensure the search feature is mobile-friendly and responsive for various device types (desktops, tablets, smartphones).\n\n#### 2.2 Feedback Mechanism\n- Provide immediate feedback (e.g., loading animations) while the search is processing.\n- Display a "No results found" message if the search yields no matches.\n\n#### 2.3 User Interface\n- Clean and user-friendly design for the search results page.\n- Thumbnail images for products in search results for better visibility.\n- Clear call-to-action buttons for each product (e.g., “View Details”, “Add to Cart”).\n\n### 3. Technical Requirements\n\n#### 3.1 Backend\n- **Search Algorithm**: Implement efficient search algorithms that utilize a well-structured database for fast query response times.\n- **Database**: Maintain a robust database that supports various attributes for products, allowing flexible searching and filtering.\n\n#### 3.2 Integration\n- **Search API**: Develop APIs for search and filter functionalities that can be integrated with the frontend of the website for seamless operation.\n- **Analytics**: Track search queries and behaviors for insights into user preferences and to improve the search functionality over time.\n\n### 4. Security and Performance Requirements\n- **Data Protection**: Ensure secure communication if using user accounts for personalized search histories or recommendations.\n- **Performance Optimization**: Optimize the search engine for quick load times, especially under high traffic scenarios.\n\n### 5. Testing Requirements\n- Conduct usability testing to ensure that the search feature is intuitive and user-friendly.\n- Perform performance testing to ensure swift response times, even with large datasets.\n\n### 6. Documentation and Training\n- Provide user guides or a tutorial for new users to help them understand how to use the search feature effectively.\n- Maintain technical documentation for the development team regarding the backend and frontend aspects of the search functionality.\n\nBy clearly outlining the above requirements, we can create a robust and user-friendly product search feature that enhances the overall shopping experience on the e-commerce saree website.', type='TextMessage'), TextMessage(id='6a531542-051a-431f-afeb-9d94aa9469f4', source='developer_agent', models_usage=RequestUsage(prompt_tokens=765, completion_tokens=1107), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 43, 9, 248331, tzinfo=datetime.timezone.utc), content='## API Design for Product Search Feature\n\nTo implement the product search feature for the e-commerce saree website, we will design a RESTful API that supports essential functionalities outlined in the previous requirements. Below is a proposed API structure, including endpoints, request/response formats, and implementation considerations.\n\n### 1. API Endpoints\n\n#### 1.1 Search Products\n- **Endpoint**: `GET /api/products/search`\n- **Description**: Search for products based on query string and filters.\n- **Query Parameters**:\n  - `query` (string): Search keyword(s).\n  - `category` (string, optional): Filter products by category.\n  - `fabric` (string, optional): Filter by fabric type.\n  - `color` (string, optional): Filter by color.\n  - `priceMin` (number, optional): Minimum price filter.\n  - `priceMax` (number, optional): Maximum price filter.\n  - `brand` (string, optional): Filter by brand.\n  - `size` (string, optional): Filter by size.\n  - `sort` (string, optional): Sort results (e.g., `price-asc`, `price-desc`, `newest`).\n  - `page` (number, optional): Pagination number.\n  - `limit` (number, optional): Number of results per page.\n  \n- **Example Request**:\n```\nGET /api/products/search?query=silk&category=bridal&priceMin=1000&priceMax=5000&sort=price-asc&page=1&limit=20\n```\n\n- **Response**:\n```json\n{\n  "currentPage": 1,\n  "totalPages": 5,\n  "totalResults": 100,\n  "products": [\n    {\n      "id": "12345",\n      "name": "Elegant Silk Saree",\n      "category": "Bridal",\n      "fabric": "Silk",\n      "color": "Red",\n      "price": 3500,\n      "imageUrl": "https://example.com/images/saree12345.jpg",\n      "brand": "SareeWorld",\n      "size": "6.3 meters"\n    },\n    {\n      "id": "12346",\n      "name": "Designer Silk Saree",\n      "category": "Bridal",\n      "fabric": "Silk",\n      "color": "Pink",\n      "price": 4500,\n      "imageUrl": "https://example.com/images/saree12346.jpg",\n      "brand": "Fashionista",\n      "size": "6.3 meters"\n    }\n  ]\n}\n```\n\n#### 1.2 Autocomplete Search Suggestions\n- **Endpoint**: `GET /api/products/autocomplete`\n- **Description**: Provide autocomplete suggestions based on the user input.\n- **Query Parameters**:\n  - `query` (string): Partial search term for autocomplete.\n\n- **Example Request**:\n```\nGET /api/products/autocomplete?query=si\n```\n\n- **Response**:\n```json\n{\n  "suggestions": [\n    "silk saree",\n    "silk lehenga",\n    "silk dupatta"\n  ]\n}\n```\n\n### 2. Implementation Considerations\n\n#### 2.1 Database Structure\n- **Products Table**:\n  - `id`: UUID/Primary Key\n  - `name`: VARCHAR\n  - `category`: VARCHAR\n  - `fabric`: VARCHAR\n  - `color`: VARCHAR\n  - `price`: DECIMAL\n  - `image_url`: VARCHAR\n  - `brand`: VARCHAR\n  - `size`: VARCHAR\n  - `created_at`: TIMESTAMP\n  - `updated_at`: TIMESTAMP\n\n#### 2.2 Search Algorithm\n- Implement text search capabilities using SQL or a search engine like Elasticsearch:\n  - Use full-text search for keyword matches.\n  - Index relevant fields for faster searching and filtering.\n\n#### 2.3 Pagination and Sorting\n- Implement pagination logic to handle large datasets efficiently.\n- Allow sorting by various fields based on user preference.\n\n### 3. Security Measures\n- **Rate Limiting**: Apply rate limiting on API endpoints to prevent abuse.\n- **Input Validation**: Validate all incoming data on the server to prevent SQL injection and other attacks.\n- **HTTPS**: Ensure the API is served over HTTPS to protect data in transit.\n\n### 4. Performance Optimization\n- Use caching mechanisms (e.g., Redis, Memcached) to cache search results for frequently searched terms to improve response time.\n- Optimize database queries to ensure they run efficiently and scale as the data grows.\n\n### 5. Testing Strategy\n- **Unit Testing**: Test individual components and functions of the API.\n- **Integration Testing**: Verify that the API integrates smoothly with the frontend and performs correctly across different scenarios.\n- **Load Testing**: Test how the API handles a high volume of requests.\n\n### 6. Documentation\n- **Swagger/OpenAPI**: Generate API documentation automatically for better reference by the development team and external clients.\n- **User Guides**: Provide concise documentation for end-users on how to utilize the search feature effectively.\n\nThis API design provides a robust and flexible approach to implementing the product search functionality for a saree e-commerce platform, aligning closely with the requirements identified earlier.', type='TextMessage'), TextMessage(id='e4fc6e82-64b1-4520-ac91-6886ec0f6654', source='qa_agent', models_usage=RequestUsage(prompt_tokens=1883, completion_tokens=920), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 43, 21, 652278, tzinfo=datetime.timezone.utc), content='Based on the provided API design and requirements for the product search feature on the e-commerce saree website, we can create functional, API, and edge test cases to ensure the search functionality operates as intended. Here’s a comprehensive set of test cases categorized accordingly:\n\n### 1. Functional Test Cases\n\n#### 1.1 Search Functionality\n- **TC-001**: Verify that entering a valid keyword ("silk saree") in the search bar returns relevant products.\n- **TC-002**: Ensure that the search returns no products when an invalid keyword ("xxx") is entered.\n- **TC-003**: Test the accuracy of the search results against the expected list of sarees.\n- **TC-004**: Check that the search bar presents the autocomplete suggestions based on the entered characters (e.g., "si").\n- **TC-005**: Confirm that the “No results found” message is displayed when the search yields no matches.\n\n#### 1.2 Filters and Sorting\n- **TC-006**: Verify that applying a category filter (e.g., "Bridal") narrows down results to only that category.\n- **TC-007**: Check that multiple filters (e.g., category, fabric) applied together yield correct results.\n- **TC-008**: Validate that products are sorted correctly when sorted by price ascending/descending.\n- **TC-009**: Test that the pagination works correctly, showing the right number of items per page and navigating between pages.\n\n#### 1.3 UI/UX\n- **TC-010**: Ensure that the search results page displays product images, names, and prices correctly.\n- **TC-011**: Verify that the UI is responsive and looks correct on different devices (desktop, tablet, mobile).\n- **TC-012**: Confirm that the loading animation appears while the search results are being processed.\n\n### 2. API Test Cases\n\n#### 2.1 Search Products Endpoint\n- **TC-013**: Send a request to the `GET /api/products/search` endpoint with valid parameters and check for a successful response (HTTP 200).\n- **TC-014**: Test the endpoint with missing required query parameter (`query`) to verify that it returns an error (HTTP 400).\n- **TC-015**: Verify that the search request with invalid filter parameters (e.g., negative price) returns appropriate error messages.\n- **TC-016**: Check pagination by sending a request with the `page` and `limit` parameters and confirm that the response includes pagination details.\n- **TC-017**: Test sort functionality by ordering results correctly based on varying parameters (e.g., `sort=price-asc`).\n\n#### 2.2 Autocomplete Suggestions Endpoint\n- **TC-018**: Validate that the `GET /api/products/autocomplete` endpoint returns appropriate suggestions for a valid input (e.g., "si").\n- **TC-019**: Verify that the endpoint returns an empty suggestions list for an input that has no matches (e.g., "xxy").\n- **TC-020**: Test the API response time for valid autocomplete queries to ensure it meets performance standards.\n\n### 3. Edge Test Cases\n\n#### 3.1 Overload and Performance\n- **TC-021**: Perform load testing by sending a high volume of requests to the search API to check its performance and stability.\n- **TC-022**: Test the effects of rapid successive searches (rate limiting) to ensure that the API does not crash or degrade performance.\n\n#### 3.2 Boundary Values\n- **TC-023**: Send search requests with maximum character limits (e.g., extremely long query string) to verify how the API handles them.\n- **TC-024**: Test the search with extreme price range values (e.g., $0 - $1,000,000) to see if the filtering process is robust.\n\n#### 3.3 Security\n- **TC-025**: Try to perform SQL injection through the search query to ensure that the application is protected against such attacks.\n- **TC-026**: Verify that the API endpoints do not expose sensitive information or internal server errors on invalid requests.\n\nThese test cases can serve as a foundational guideline to validate the search functionality comprehensively, ensuring both functional correctness and robustness in various scenarios. Leveraging automation tools such as Selenium for functional testing and Postman or Swagger for API testing can facilitate the execution of these cases effectively.', type='TextMessage'), TextMessage(id='4c5d190f-50ca-4780-a8d6-b65c9b6e21dd', source='reviewer_agent', models_usage=RequestUsage(prompt_tokens=2807, completion_tokens=520), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 43, 30, 93288, tzinfo=datetime.timezone.utc), content='The provided API design and testing strategy are well thought out, but there are several gaps and areas needing clarification or improvement:\n\n### Gaps and Improvements Needed:\n\n1. **Functional Test Cases**:\n   - **TC-003**: There should be a specific set of expected results or criteria outlined for accuracy verification. Tests need detailed expected results for validation.\n   - **TC-004**: The test case does not mention verifying the content of autocomplete suggestions against expected results.\n   - **TC-007**: This test case should specify what combinations of filters are being tested, e.g., "Bridal" category with "Silk" fabric.\n\n2. **API Test Cases**:\n   - **TC-014**: Specify what the error message content should be for clarity on handling missing parameters.\n   - **TC-016**: Details on the expected pagination should include what page numbers and counts are anticipated. Clarify bounds for valid limits (e.g., 1-100).\n   - **TC-018**: Include expected number of suggestions when checking autocomplete suggestions.\n\n3. **Edge Test Cases**:\n   - **TC-021**: The expected performance metrics (response times, error rate thresholds) for load testing should be defined.\n   - **TC-024**: Expect the API to define valid and invalid boundaries (e.g., the maximum number of products that can be passed as filters).\n\n4. **Security Test Cases**:\n   - **TC-025**: Expand to include other common web vulnerabilities, such as XSS and CSRF, beyond just SQL injection.\n   - **TC-026**: Specify the criteria for what constitutes "sensitive information" or what "internal server errors" should be avoided.\n\n5. **Documentation**:\n   - There is no mention in the testing documentation of how the results from these tests will be recorded or tracked. Suggest implementing a tracking or reporting mechanism.\n\n### Suggested Enhancements:\n- Consider adding a test case for verifying error handling for unsupported or invalid sorts in the API.\n- Include tests related to user experience aspects like loading times and responsiveness beyond just the presence of loading indicators.\n- Introduce a test for assessing the browser compatibility of the search feature on different web browsers.\n- Provide guidance on the expected method for measuring performance in test cases related to API response times (for example, using specific performance testing tools).\n  \n### Approval Status:\n**Not yet approved.** The testing plan requires enhancement in details, coverage, and clear criteria for validation metrics before it can be considered complete and ready for implementation.', type='TextMessage'), TextMessage(id='9ba130cf-9d4e-455b-9758-1f8953f6ae8d', source='planner_agent', models_usage=RequestUsage(prompt_tokens=3331, completion_tokens=1029), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 43, 44, 125133, tzinfo=datetime.timezone.utc), content='Thank you for your thorough review and insightful feedback on the API design and testing strategy for the product search feature. Here’s a plan to address the gaps and suggestions for enhancements:\n\n### Revised Test Case Strategy\n\n#### 1. Functional Test Cases\n\n- **TC-001**: Verify that entering a valid keyword ("silk saree") in the search bar returns relevant products.\n- **TC-002**: Ensure that the search returns no products when an invalid keyword ("xxx") is entered.\n- **TC-003**: **(Revised)** Test the accuracy of the search results against a defined expected list of sarees. Example: Given "silk saree", expect products with `id: 12345` and `id: 12346` based on predefined fixtures.\n- **TC-004**: **(Revised)** Check that the autocomplete suggestions for "si" return expected results: ["silk saree", "silk lehenga", "silk dupatta"].\n- **TC-005**: Confirm display of the "No results found" message when no matches are found.\n- **TC-006**: Verify that applying a category filter (e.g., "Bridal") returns only products in the "Bridal" category.\n- **TC-007**: **(Revised)** Check multiple filters together: "Bridal" (category) and "Silk" (fabric) to ensure only those products are returned.\n- **TC-008**: Validate correct sorting by price (ascending and descending) meets user expectations.\n- **TC-009**: Check pagination shows correct items per page with logical navigation between pages based on total items.\n\n#### 2. API Test Cases\n\n- **TC-013**: Verify `GET /api/products/search` with valid parameters returns a successful response.\n- **TC-014**: **(Revised)** Test the endpoint with a missing required query parameter (`query`) to ensure it returns a 400 error and specifies the message: "Missing required query parameter: query".\n- **TC-015**: Verify that submitting invalid filter parameters (e.g., negative price) returns appropriate error messages, such as "Invalid filter parameter: price".\n- **TC-016**: **(Revised)** Request with `page` and `limit` parameters should include paginated response with defined expectations: `{ "currentPage": 1, "totalPages": X, "totalResults": Y }` and limits should be between 1-100.\n- **TC-017**: Test sort functionality to validate results are ordered correctly for different parameters (e.g., `sort=price-asc`).\n- **TC-018**: **(Revised)** Validate that the `GET /api/products/autocomplete` returns expected suggestions for input (e.g., "si") with expected number of suggestions (3).\n- **TC-019**: Verify that the endpoint returns an empty suggestions list for no matches (e.g., "xxy").\n- **TC-020**: Test API response time for valid autocomplete queries to ensure response time is less than 200ms.\n\n#### 3. Edge Test Cases\n\n- **TC-021**: **(Revised)** Conduct load testing with expected performance metrics: response times < 500ms under load and error rates < 1%.\n- **TC-022**: Validate API performs within acceptable limits with rapid successive searches without degradation.\n- **TC-023**: Assess handling of maximum character limits in search queries (e.g., 255 characters) to verify system robustness.\n- **TC-024**: **(Revised)** Test extreme price ranges with valid boundaries set (e.g., $0 to $1,000,000 and beyond). Confirm filtering is correctly implemented.\n  \n#### 4. Security Test Cases\n\n- **TC-025**: **(Expanded)** Test against a broader set of vulnerabilities, including SQL Injection, XSS, and CSRF, verifying appropriate response.\n- **TC-026**: **(Expanded)** Ensure no sensitive information (e.g., user data, internal server paths) is exposed in error messages or API responses.\n\n### Additional Suggestions\n\n- **Tracking Mechanism**: Implement a systematic approach for recording and tracking test results using a testing tool or a custom dashboard where tests can be marked as passed/failed.\n- **User Experience Tests**: Add tests to verify load times beyond loading indicators—establish expected maximum wait times for search results and ensure responsiveness on various devices/browsers.\n- **Browser Compatibility Test**: Introduce tests that verify the search feature on major browsers, ensuring consistency in design and functionality.\n- **Performance Testing Tools**: Suggest using tools like JMeter or LoadRunner for assessing API performance and benchmarking response times.\n\n### Approved Status Update\n\nAfter addressing these gaps and suggestions for enhancements, I will seek to submit the updated testing documentation for approval. Thank you for your detailed feedback, and I appreciate the opportunity to refine this strategy for the product search feature.', type='TextMessage'), TextMessage(id='829e6a6c-8dc9-4eee-b95a-6cebc4a93acf', source='developer_agent', models_usage=RequestUsage(prompt_tokens=4367, completion_tokens=545), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 43, 54, 266593, tzinfo=datetime.timezone.utc), content='Your revised test case strategy effectively addresses the gaps identified in the initial set of test cases while also incorporating the suggestions for enhancements. The adjustments for clarity and detail will significantly improve the robustness and comprehensiveness of the testing approach for the product search feature on the e-commerce saree website. Here are a few final thoughts and additional suggestions that could further enhance your testing strategy:\n\n### Final Suggestions\n\n1. **Maintainability of Test Cases**:\n   - Consider versioning the test cases to manage changes over time. This will allow you to keep track of updates, especially when features are modified or when new features are added to the search functionality.\n\n2. **Automation**:\n   - Implement automation for as many test cases as possible, especially API and functional tests. Tools like Postman (for API testing) and Selenium or Cypress (for UI testing) can streamline testing processes and facilitate continuous integration/continuous deployment (CI/CD) practices.\n\n3. **Data Management**:\n   - Create a set of dummy data to populate the database for testing purposes. This will allow for consistent and reproducible test conditions. Ensure that this data aligns with the various edge cases you want to test.\n\n4. **User Scenarios**:\n   - Incorporate real-world user scenarios in your testing, such as:\n     - Typical search patterns (users searching for a specific occasion).\n     - Searches based on upcoming festivals or events (e.g., Diwali saree shopping).\n     - Different user personas who may have varying levels of familiarity with the site (first-time users vs. returning users).\n\n5. **Accessibility Testing**:\n   - Consider including tests related to accessibility, ensuring that the search features meet WCAG (Web Content Accessibility Guidelines) standards. This would involve checking that the search bar is navigable via keyboard, screen readers can identify it, and color contrasts are acceptable.\n\n6. **Performance Metrics**:\n   - Document expectations for performance under different conditions (normal load, peak load) within the testing strategy. Establish clear benchmarks and ensure these are communicated to the stakeholders.\n\n7. **User Acceptance Testing (UAT)**:\n   - After internal testing, plan for User Acceptance Testing where actual users verify that the search functionality meets their expectations and needs. Gather feedback from them to make further enhancements.\n\n### Monitoring and Feedback\nEnsure that there is a mechanism for gathering ongoing feedback from users after the product launch, which can help identify issues that may not have been caught during testing.\n\n### Approval Status:\nOnce you have incorporated these suggestions, the revised documentation and testing strategy should be well-prepared for re-submission for approval. Excellent work on enhancing the strategy while maintaining focus on comprehensive test coverage and quality assurance for the product search feature.', type='TextMessage'), TextMessage(id='9bddd85b-87b0-479a-ba51-347cbe8fb0c4', source='qa_agent', models_usage=RequestUsage(prompt_tokens=4923, completion_tokens=617), metadata={}, created_at=datetime.datetime(2026, 6, 5, 19, 44, 3, 729140, tzinfo=datetime.timezone.utc), content='Thank you for your valuable final suggestions and supportive feedback on the revised test case strategy. Your recommendations will certainly enhance the testing approach for the product search feature. Here’s how we’ll implement these suggestions into our existing strategy:\n\n### Action Plan for Enhancements\n\n#### 1. Maintainability of Test Cases\n- **Versioning System**: Introduce a versioning system for all test cases with clear changelogs. Each test case will have a unique identifier and version number to track updates over time.\n\n#### 2. Automation\n- **Automated Testing**: Begin identifying key areas of the test cases that can be automated. Set up a framework using:\n  - **Postman** for API testing, enabling automated regression tests.\n  - **Selenium/Cypress** for UI testing to automate functional tests related to the search feature.\n- **CI/CD Integration**: Integrate automated tests within the CI/CD pipeline to run on every build to ensure that new changes do not break existing functionality.\n\n#### 3. Data Management\n- **Dummy Data Preparation**: Develop a strategy for creating and maintaining a robust set of dummy data that can be used consistently during testing. This will include a database seeding process that can be executed prior to running tests.\n\n#### 4. User Scenarios\n- **Real-World User Scenarios**: Capture various user scenarios in the test cases based on user persona studies. This will guide the creation of tests reflective of typical user behavior and seasonal trends (like festival shopping).\n  \n#### 5. Accessibility Testing\n- **Incorporating Accessibility Checks**: Establish accessibility testing in both functional and UI tests using tools like Axe or Lighthouse to ensure compliance with WCAG standards. Create test cases specifically targeting keyboard navigation and screen reader compatibility.\n\n#### 6. Performance Metrics\n- **Documenting Performance Expectations**: Explicitly outline performance benchmarks for normal and peak load scenarios. This will set clear expectations for acceptable response times and other performance indicators during testing.\n\n#### 7. User Acceptance Testing (UAT)\n- **User Involvement in Testing**: Develop a UAT plan following internal testing. Engage real users to validate the search feature, collect feedback, and make adjustments based on their input.\n\n### Monitoring and Feedback\n- **Feedback Mechanism**: Set up a process for ongoing user feedback post-launch through tools like surveys or feedback forms that can be analyzed to capture any recurring issues or suggestions for enhancements.\n\n### Conclusion\nBy strategically incorporating these enhancements, we can ensure that the testing strategy for the product search feature is comprehensive, maintainable, and adaptable to meet both the developers’ and users’ needs. The ultimate goal is to deliver a high-quality, user-oriented search feature that aligns with the business objectives of the e-commerce saree website.\n\nWith these adjustments, the testing strategy is fully prepared for re-submission for approval, with the intent to deliver a robust product search feature that excels in functionality, usability, and performance. Thank you once more for your detailed insights, which have significantly shaped the direction of this project.

## 12. Real-world program: Multi-agent test case generator

Scenario: You are an SDET/Test Manager. You want agents to generate test cases from a user story.

### User story

> As a customer, I want to filter sarees by fabric, color, price, and availability so that I can quickly find products I like.

### Agents

| Agent | Responsibility |
|---|---|
| RequirementAgent | Extracts acceptance criteria |
| QAAgent | Generates test cases |
| AutomationAgent | Suggests automation approach |
| ReviewAgent | Checks missing scenarios |

In [26]:
USER_STORY = """
As a customer, I want to filter sarees by fabric, color, price, and availability
so that I can quickly find products I like.
"""

def requirement_agent(story: str) -> str:
    return """
Acceptance Criteria:
1. User can filter by fabric: cotton, silk, chiffon, georgette.
2. User can filter by color.
3. User can filter by minimum and maximum price.
4. User can filter only available products.
5. Multiple filters can be applied together.
6. User can clear filters.
7. Filter result count should update correctly.
""".strip()

def qa_agent(acceptance_criteria: str) -> str:
    return """
Functional Test Cases:
1. Apply fabric filter and verify only selected fabric products are shown.
2. Apply color filter and verify product colors match.
3. Apply price range and verify all products are within range.
4. Enable available-only filter and verify out-of-stock products are hidden.
5. Apply fabric + color + price filters together.
6. Clear filters and verify default product listing returns.
7. Apply filters that return no products and verify empty-state message.
8. Refresh page and verify expected filter persistence behavior.
""".strip()

def automation_agent(test_cases: str) -> str:
    return """
Automation Approach:
- Use Playwright/WebDriverIO for UI tests.
- Use API tests to validate backend filter response.
- Keep test data fixed for predictable validation.
- Validate product cards using attributes: fabric, color, price, stock.
- Add visual checks only for critical UI elements, not every product image.
""".strip()

def review_agent(all_outputs: str) -> str:
    return """
Review Gaps:
- Add performance test for large catalog.
- Add mobile responsive validation.
- Add invalid price range test, e.g., min_price greater than max_price.
- Add accessibility checks for filter controls.
- Add analytics event validation if business tracks filter usage.
""".strip()

acceptance = requirement_agent(USER_STORY)
tests = qa_agent(acceptance)
automation = automation_agent(tests)
review = review_agent(acceptance + tests + automation)

print("USER STORY:\n", USER_STORY)
print("\n--- RequirementAgent ---\n", acceptance)
print("\n--- QAAgent ---\n", tests)
print("\n--- AutomationAgent ---\n", automation)
print("\n--- ReviewAgent ---\n", review)

USER STORY:
 
As a customer, I want to filter sarees by fabric, color, price, and availability
so that I can quickly find products I like.


--- RequirementAgent ---
 Acceptance Criteria:
1. User can filter by fabric: cotton, silk, chiffon, georgette.
2. User can filter by color.
3. User can filter by minimum and maximum price.
4. User can filter only available products.
5. Multiple filters can be applied together.
6. User can clear filters.
7. Filter result count should update correctly.

--- QAAgent ---
 Functional Test Cases:
1. Apply fabric filter and verify only selected fabric products are shown.
2. Apply color filter and verify product colors match.
3. Apply price range and verify all products are within range.
4. Enable available-only filter and verify out-of-stock products are hidden.
5. Apply fabric + color + price filters together.
6. Clear filters and verify default product listing returns.
7. Apply filters that return no products and verify empty-state message.
8. Refresh 

## 13. Real-world program: Multi-agent SQL validation workflow

Scenario: You are validating business metrics such as `TotalSpend` and `OrderRevenue`.

Agents:

| Agent | Role |
|---|---|
| DataUnderstandingAgent | Understands columns and formulas |
| SQLAgent | Writes SQL validation query |
| QAAgent | Defines pass/fail rules |
| ReviewerAgent | Checks edge cases |

This is similar to real enterprise data testing workflows.

In [28]:
metric_context = {
    "table": "E2E_FACT_SUSTAINABILITY_METRICS",
    "formulae": {
        "TotalSpend": "QUANTITY * SOURCINGUNITCOST",
        "OrderRevenue": "QUANTITY * LISTPRICE"
    },
    "keys": ["ITEM", "LOCATION", "SHIPFROMLOCATION", "SUPPLYMETHOD", "SUPPLYDATE"]
}

def data_understanding_agent(context):
    return f"""
We need to validate metrics in table {context['table']}.
Business formulas:
- TotalSpend = {context['formulae']['TotalSpend']}
- OrderRevenue = {context['formulae']['OrderRevenue']}
Comparison should happen at key level: {', '.join(context['keys'])}.
""".strip()

def sql_agent(context):
    keys = ", ".join(context["keys"])
    return f"""
SELECT
  {keys},
  SUM(TotalSpend) AS actual_total_spend,
  SUM(QUANTITY * SOURCINGUNITCOST) AS expected_total_spend,
  SUM(OrderRevenue) AS actual_order_revenue,
  SUM(QUANTITY * LISTPRICE) AS expected_order_revenue,
  CASE
    WHEN ABS(SUM(TotalSpend) - SUM(QUANTITY * SOURCINGUNITCOST)) < 0.0001
     AND ABS(SUM(OrderRevenue) - SUM(QUANTITY * LISTPRICE)) < 0.0001
    THEN 'PASS'
    ELSE 'FAIL'
  END AS validation_status
FROM {context['table']}
GROUP BY {keys};
""".strip()

def qa_rule_agent():
    return """
Pass/Fail Rules:
- PASS if difference is less than tolerance 0.0001.
- FAIL if actual and expected values differ beyond tolerance.
- Separately track null values in quantity, cost, and list price.
- Validate row counts before comparing sums.
""".strip()

def sql_reviewer_agent():
    return """
Review Suggestions:
- Use COALESCE if null values are expected.
- Confirm currency conversion before validating spend.
- Confirm whether aggregation should be weekly, monthly, or daily.
- Confirm if duplicate planned supply records need deduplication before comparison.
""".strip()

print("--- DataUnderstandingAgent ---")
print(data_understanding_agent(metric_context))
print("\n--- SQLAgent ---")
print(sql_agent(metric_context))
print("\n--- QAAgent ---")
print(qa_rule_agent())
print("\n--- ReviewerAgent ---")
print(sql_reviewer_agent())

--- DataUnderstandingAgent ---
We need to validate metrics in table E2E_FACT_SUSTAINABILITY_METRICS.
Business formulas:
- TotalSpend = QUANTITY * SOURCINGUNITCOST
- OrderRevenue = QUANTITY * LISTPRICE
Comparison should happen at key level: ITEM, LOCATION, SHIPFROMLOCATION, SUPPLYMETHOD, SUPPLYDATE.

--- SQLAgent ---
SELECT
  ITEM, LOCATION, SHIPFROMLOCATION, SUPPLYMETHOD, SUPPLYDATE,
  SUM(TotalSpend) AS actual_total_spend,
  SUM(QUANTITY * SOURCINGUNITCOST) AS expected_total_spend,
  SUM(OrderRevenue) AS actual_order_revenue,
  SUM(QUANTITY * LISTPRICE) AS expected_order_revenue,
  CASE
    WHEN ABS(SUM(TotalSpend) - SUM(QUANTITY * SOURCINGUNITCOST)) < 0.0001
     AND ABS(SUM(OrderRevenue) - SUM(QUANTITY * LISTPRICE)) < 0.0001
    THEN 'PASS'
    ELSE 'FAIL'
  END AS validation_status
FROM E2E_FACT_SUSTAINABILITY_METRICS
GROUP BY ITEM, LOCATION, SHIPFROMLOCATION, SUPPLYMETHOD, SUPPLYDATE;

--- QAAgent ---
Pass/Fail Rules:
- PASS if difference is less than tolerance 0.0001.
- FAIL if a

## 14. Human-in-the-loop in AutoGen

Human-in-the-loop means the system asks a human before continuing.

Use it when:

- sending email,
- deleting records,
- deploying to production,
- approving financial decisions,
- merging code,
- executing database updates.

Example flow:

```text
PlannerAgent creates deployment plan
↓
SecurityAgent reviews risks
↓
Human approves
↓
DeploymentAgent runs deployment
```

For high-risk workflows, never allow agents to directly execute destructive actions without approval.

## 15. Termination conditions

A multi-agent workflow must know when to stop.

Common termination strategies:

| Strategy | Example |
|---|---|
| Max messages | Stop after 10 messages |
| Keyword | Stop when reviewer says `APPROVED` |
| Success condition | Stop when all tests pass |
| Human approval | Stop until user confirms |
| Timeout | Stop after N seconds/minutes |

Without termination, agents can enter loops and waste tokens.

## 16. Memory in Multi-Agent Systems

Memory helps agents remember useful context.

Types of memory:

| Type | Meaning | Example |
|---|---|---|
| Short-term memory | Current conversation | Requirements discussed in this run |
| Long-term memory | Stored facts/preferences | Preferred tech stack: FastAPI + Next.js |
| Vector memory | Semantic search over documents | Retrieve policies, specs, test plans |
| Tool memory | External system state | Jira ticket status, DB records |

### Caution
Memory should be controlled carefully. Do not store sensitive data unnecessarily.

## 17. Tools and code execution

AutoGen can integrate with tools and code executors. A common pattern is:

```text
LLM Agent proposes code
↓
Code Executor runs code in sandbox/container
↓
Result is returned to agent
↓
Agent fixes errors or explains output
```

This is useful for:

- data analysis,
- chart generation,
- test data creation,
- debugging scripts,
- validating calculations.

### Security warning
Never allow unrestricted code execution in production. Use sandboxing, network restrictions, timeouts, and approval gates.

## 18. How to design a good AutoGen multi-agent application

Use this checklist:

### Step 1: Define the business goal
Bad:

> Build agents for testing.

Good:

> Given a user story, generate acceptance criteria, test cases, automation approach, and review gaps.

### Step 2: Decide agent roles
Keep roles clear and non-overlapping.

Example:

- RequirementAgent
- QAAgent
- AutomationAgent
- SecurityAgent
- ReviewAgent

### Step 3: Decide workflow pattern

- Sequential if steps are fixed.
- Round-robin if agents need discussion.
- Manager-based if task routing is dynamic.

### Step 4: Add tools only where needed
Do not give every agent every tool.

### Step 5: Add termination
Always define stop rules.

### Step 6: Add observability
Log:

- agent name,
- input message,
- output message,
- tool calls,
- errors,
- token usage,
- final decision.

### Step 7: Evaluate output
Use test datasets and scoring criteria.

## 19. Common interview-style questions and answers

### Q1. What is a multi-agent system?
A system where multiple specialized agents collaborate to complete a task. In GenAI, agents are usually LLM-powered components that communicate, reason, use tools, and produce outputs.

### Q2. Why not use a single agent?
A single agent may miss details in complex tasks. Multiple agents allow specialization, review, and better task decomposition.

### Q3. What is AutoGen used for?
AutoGen is used to build conversational and collaborative AI agents that can work together, use tools, execute code, and solve complex workflows.

### Q4. What is a round-robin group chat?
It is a pattern where agents respond one by one in fixed order until a termination condition is met.

### Q5. What is human-in-the-loop?
It means the workflow pauses for human input or approval before continuing, especially for risky actions.

### Q6. What are common risks in multi-agent systems?
High cost, latency, infinite loops, hallucination, poor coordination, tool misuse, and security issues.

### Q7. How do you make multi-agent systems reliable?
Use clear roles, strict prompts, tool validation, termination conditions, logging, test cases, human approvals, and evaluation metrics.

## 20. Mini project idea for your GenAI course

### Project: AutoGen Test Case Generator

Input:

```text
User story or requirement document
```

Agents:

1. RequirementAgent extracts acceptance criteria.
2. QAAgent creates test cases.
3. AutomationAgent suggests WebDriverIO/Playwright/API automation.
4. SecurityAgent adds security test cases.
5. ReviewAgent gives final review.

Output:

- Acceptance criteria
- Functional test cases
- API test cases
- Negative test cases
- Security test cases
- Automation strategy
- Review comments

### Enhancement ideas

- Export output to Excel.
- Create Jira test cases automatically.
- Generate Cucumber feature file.
- Generate WebDriverIO step definition skeleton.
- Add human approval before Jira creation.

In [36]:
# Mini project starter: Generate a Cucumber feature skeleton from test cases

def generate_cucumber_feature(feature_name: str, scenarios: list[str]) -> str:
    lines = [f"Feature: {feature_name}", ""]
    for i, scenario in enumerate(scenarios, start=1):
        lines.append(f"  Scenario: {scenario}")
        lines.append("    Given the user is on the product listing page")
        lines.append("    When the user applies the required filters")
        lines.append("    Then the product results should match the selected filters")
        lines.append("")
    return "\n".join(lines)

scenarios = [
    "Filter sarees by fabric",
    "Filter sarees by color",
    "Filter sarees by price range",
    "Filter only available products",
    "Clear all selected filters"
]

feature_text = generate_cucumber_feature("Product Filter", scenarios)
print(feature_text)

Feature: Product Filter

  Scenario: Filter sarees by fabric
    Given the user is on the product listing page
    When the user applies the required filters
    Then the product results should match the selected filters

  Scenario: Filter sarees by color
    Given the user is on the product listing page
    When the user applies the required filters
    Then the product results should match the selected filters

  Scenario: Filter sarees by price range
    Given the user is on the product listing page
    When the user applies the required filters
    Then the product results should match the selected filters

  Scenario: Filter only available products
    Given the user is on the product listing page
    When the user applies the required filters
    Then the product results should match the selected filters

  Scenario: Clear all selected filters
    Given the user is on the product listing page
    When the user applies the required filters
    Then the product results should matc

## 21. Best practices summary

| Area | Best Practice |
|---|---|
| Agent design | Keep roles specific |
| Prompting | Give clear responsibility and output format |
| Tools | Give minimum required tool access |
| Safety | Add human approval for risky actions |
| Reliability | Use critic/reviewer agents |
| Cost | Limit message rounds and agents |
| Debugging | Log messages and tool calls |
| Evaluation | Compare output against expected criteria |
| Security | Sandbox code execution and protect secrets |

## 22. Common mistakes to avoid

1. Creating too many agents without a clear purpose.
2. Giving all tools to all agents.
3. Not setting a termination condition.
4. Not validating tool outputs.
5. Allowing agents to perform destructive actions without approval.
6. Using old AutoGen code with new AutoGen packages without checking version compatibility.
7. Not logging intermediate agent messages.
8. Treating agent output as always correct.

## 23. Quick revision notes

- Multi-agent systems split complex tasks across specialized agents.
- AutoGen helps create conversational agents and teams.
- Common patterns: sequential, round-robin, manager/supervisor, critic, tool-specialist.
- Tools make agents practical by connecting them to real systems.
- Termination conditions are mandatory.
- Human approval is important for risky workflows.
- Evaluation and logging are needed for production readiness.

## 24. References for further study

- AutoGen official documentation: https://microsoft.github.io/autogen/stable/
- AutoGen AgentChat guide: https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/index.html
- AutoGen GitHub repository: https://github.com/microsoft/autogen
- Microsoft Research AutoGen project: https://www.microsoft.com/en-us/research/project/autogen/
- AutoGen paper: https://www.microsoft.com/en-us/research/publication/autogen-enabling-next-gen-llm-applications-via-multi-agent-conversation-framework/